# Topic 19: Python Skills for GenAI

**Topics:** Large data · Embeddings · Text preprocessing · Tokenization · Bag of Words · TF-IDF · AI libraries


## 1. Working with Large Datasets Efficiently


In [ ]:
# Generator: yields one item at a time — low memory
def data_generator(n):
    for i in range(n):
        yield {"id": i, "text": f"Document {i}", "label": i % 2}

total, counts = 0, {0:0, 1:0}
for rec in data_generator(1_000_000):
    counts[rec["label"]] += 1
    total += 1

print(f"Processed {total:,} records")
print(f"Class distribution: {counts}")

## 2. Understanding Embeddings

An **embedding** is a list of numbers (vector) that represents a word or sentence.
Similar meanings → similar vectors.

Famous example: `king - man + woman ≈ queen`


In [ ]:
import numpy as np

embeddings = {
    "king":  np.array([0.82, 0.14, -0.31]),
    "queen": np.array([0.79, 0.18, -0.28]),
    "man":   np.array([0.65, 0.05, -0.40]),
    "woman": np.array([0.62, 0.09, -0.37]),
    "car":   np.array([0.02, -0.55, 0.80]),
}

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similarity:")
for w1, w2 in [("king","queen"),("king","car"),("man","woman")]:
    print(f"  {w1:6} ↔ {w2:6}: {cosine_sim(embeddings[w1],embeddings[w2]):.3f}")

# king - man + woman = ?
result = embeddings["king"] - embeddings["man"] + embeddings["woman"]
print("\nking - man + woman similarity to each word:")
for w, v in embeddings.items():
    print(f"  {w}: {cosine_sim(result,v):.3f}")

## 3. Text Preprocessing


In [ ]:
import re

texts = [
    "The quick brown fox jumps over the lazy dog!",
    "Machine Learning and AI are transforming the world.",
    "Python is a GREAT language for Data Science & GenAI.",
]

STOPWORDS = {"the","is","a","an","and","are","to","for","of","in","it","be","with","was"}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    tokens = [w for w in text.split() if w not in STOPWORDS]
    return tokens

for t in texts:
    print(f"Original : {t}")
    print(f"Processed: {preprocess(t)}")
    print()

## 4. Tokenization


In [ ]:
from collections import Counter
import re

def tokenize(text):
    return re.sub(r"[^a-zA-Z0-9\s]"," ",text).lower().split()

all_docs = [
    "I love machine learning",
    "machine learning is fun",
    "I love Python for machine learning",
    "deep learning is part of machine learning",
]

all_tokens = [t for doc in all_docs for t in tokenize(doc)]
vocab = sorted(set(all_tokens))
word2id = {w:i for i,w in enumerate(vocab)}

print(f"Vocabulary ({len(vocab)} words): {vocab[:10]}...")

sample = "machine learning is great"
ids = [word2id.get(w, -1) for w in sample.split()]
print(f"\nText  : {sample}")
print(f"IDs   : {ids}")

print("\nTop 10 words:")
for w, c in Counter(all_tokens).most_common(10):
    print(f"  {w!r}: {c}")

## 5. Bag of Words & TF-IDF


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

docs = [
    "I love machine learning",
    "machine learning is fun",
    "I love Python for machine learning",
    "deep learning is part of machine learning",
]

# Bag of Words
cv = CountVectorizer()
bow = pd.DataFrame(cv.fit_transform(docs).toarray(), columns=cv.get_feature_names_out())
print("Bag of Words:")
print(bow)

# TF-IDF
tfidf = TfidfVectorizer()
tf = pd.DataFrame(tfidf.fit_transform(docs).toarray().round(3),
                   columns=tfidf.get_feature_names_out())
print("\nTF-IDF:")
print(tf)

## 6. GenAI API Structure (Reference)

You do **not** need to train your own model. Call a powerful model via API:

```python
import anthropic
client = anthropic.Anthropic(api_key="your-api-key")

response = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Explain embeddings in 2 sentences."}]
)
print(response.content[0].text)
```

Install with: `pip install anthropic`


## ✅ Summary

| Concept | Key Point |
|---|---|
| Generator | Processes data one item at a time — saves RAM |
| Embedding | Dense vector representing meaning |
| Cosine similarity | Measure closeness of embeddings |
| Text preprocessing | Lowercase → remove punctuation → remove stopwords |
| Tokenisation | Split text into tokens (word/subword/char) |
| Bag of Words | Count of each word per document |
| TF-IDF | Weighted counts — boosts rare, specific words |
